# Folder 01 / file 09 — promotion_ready (after Compare; sets @champion on go)

UCI Adult Census Income ([dataset](https://archive.ics.uci.edu/dataset/2/adult)): binary target `income_gt_50k` where **`>50K` = 1** and **`<=50K` = 0**. Fourteen census features; official split is `adult.data` (train) / `adult.test` (holdout).

Inlined replica of `src/n01_dev_train/n09_promotion_ready.py`. Run cells **in order** (local or Jobs). Catalog/schema/model/mode come from task env.

Mark a go Adult model promotion-ready (sets ml_dev @champion on go only).


## 1 — Imports


In [ ]:
STOP = False

def _stop(msg: str = "") -> None:
    global STOP
    STOP = True
    print(msg)
    try:
        dbutils.notebook.exit(msg or "ok")  # noqa: F821
    except Exception:
        pass

from src.n00_shared.runtime import (
    Settings,
    assert_dev_ml_allowed,
    configure_mlflow,
    get_task_value,
    job_run_id,
    load_settings,
    mlflow_client,
    set_task_value,
)


## 2 — `_tag_map`


In [ ]:
def _tag_map(client, name: str, version: str) -> dict:
    mv = client.get_model_version(name, version)
    tags = mv.tags or {}
    if isinstance(tags, dict):
        return tags
    return {t.key: t.value for t in tags}


## 3 — `settings = load_settings()`


In [ ]:
settings = load_settings()


## 4 — `run()` step 1/1


In [ ]:
if not STOP:
    assert_dev_ml_allowed(settings)
    configure_mlflow(settings)
    client = mlflow_client()
    version = get_task_value("compare", "model_version")
    tags = _tag_map(client, settings.source_model_name, version)
    go = str(tags.get("compare_result") or "") == "go"
    if go:
        client.set_registered_model_alias(settings.source_model_name, "champion", version)
        client.set_model_version_tag(settings.source_model_name, version, "ready", "true")
        client.set_model_version_tag(settings.source_model_name, version, "go_version", str(version))
        run_id = job_run_id()
        client.set_model_version_tag(settings.source_model_name, version, "train_run_id", run_id)
        set_task_value("ready", "true")
        set_task_value("go_version", str(version))
        set_task_value("train_run_id", run_id)
        print(f"promotion_ready go version={version}")
        _stop()
    client.set_model_version_tag(settings.source_model_name, version, "ready", "false")
    set_task_value("ready", "false")
    print("promotion_ready no-go; champion unchanged; job succeeds")
